# 案例一：豆瓣图书爬取与 MySQL 存储

爬取豆瓣新书速递页面中的图书信息，并将书名、评分、作者、出版社、出版日期和简介保存到 MySQL 的 `bookinfo` 表中。代码按照教材的 `get_html()`、`parser()`、`save_mysql()` 和主程序四部分组织。

## 一、用到的知识点

- **Requests**：发送 GET 请求，获取网页 HTML。
- **XPath**：定位图书信息所在的 HTML 标签并提取文本。
- **PyMySQL**：连接 MySQL，执行 SQL 语句。
- **批量插入**：`executemany()` 可以一次写入多条记录；`commit()` 提交事务，发生异常时用 `rollback()` 回滚。
- **参数化 SQL**：使用 `%s` 作为占位符，数据单独传入，避免直接拼接 SQL。

In [ ]:
%pip install requests lxml pymysql

In [1]:
import requests
from lxml import etree
import pymysql

## 二、数据库配置

先创建 `spider` 数据库，再在其中创建 `bookinfo` 表。字段名称和教材保持一致。

In [2]:
dbinfo = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "root",
    "password": "123456",
    "charset": "utf8mb4",
    "cursorclass": pymysql.cursors.DictCursor
}

# 创建数据库
conn = pymysql.connect(
    host=dbinfo["host"],
    port=dbinfo["port"],
    user=dbinfo["user"],
    password=dbinfo["password"],
    charset=dbinfo["charset"]
)
cursor = conn.cursor()
cursor.execute("CREATE DATABASE IF NOT EXISTS spider DEFAULT CHARACTER SET utf8mb4")
conn.commit()
cursor.close()
conn.close()

# 指定要连接的数据库
dbinfo["database"] = "spider"

# 创建图书信息表
conn = pymysql.connect(**dbinfo)
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS bookinfo (
    id INT NOT NULL AUTO_INCREMENT PRIMARY KEY,
    bookname VARCHAR(100) NOT NULL,
    score VARCHAR(20),
    autor VARCHAR(100),
    press VARCHAR(200),
    pubdate VARCHAR(20),
    describ VARCHAR(3000)
) DEFAULT CHARSET=utf8mb4
""")
conn.commit()
cursor.close()
conn.close()

print("数据库和数据表创建完成")

数据库和数据表创建完成


## 三、获取网页数据

使用 Requests 发送 GET 请求。设置 `User-Agent` 请求头，获取网页文本；`raise_for_status()` 用于检查请求状态。

In [3]:
def get_html(url, timeout=30):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/80.0.3987.132 Safari/537.36"
    }
    try:
        r = requests.get(url, headers=headers, timeout=timeout)
        r.encoding = r.apparent_encoding
        r.raise_for_status()
        return r.text
    except requests.RequestException as error:
        print("网页请求失败：", error)
        return None

## 四、解析图书信息

将 HTML 转换为可查询的元素对象，遍历图书所在的 `li` 标签。作者、出版社和出版日期位于同一段文本中，使用 `/` 分割后分别提取。部分图书信息可能不完整，因此对缺失字段作简单处理。

In [4]:
def parser(html):
    doc = etree.HTML(html)
    out_list = []

    for row in doc.xpath("//div[@id='content']//li"):
        title = row.xpath(".//h2/a/text()")
        score = row.xpath(".//p[@class='rating']/span[2]/text()")
        info = row.xpath(".//p[@class='color-gray']/text()")
        describe = row.xpath(".//p[3]//text()")

        if not title:
            continue

        title = title[0].strip()
        score = score[0].strip() if score else ""
        info = [item.strip() for item in info[0].split("/")] if info else []

        author = info[0] if len(info) >= 1 else ""
        press = info[-2] if len(info) >= 3 else ""
        pubdate = info[-1] if len(info) >= 3 else ""
        describe = "".join(describe).strip()

        row_list = [title, score, author, press, pubdate, describe]
        out_list.append(row_list)

    return out_list

## 五、保存到 MySQL

`executemany()` 批量执行插入语句。插入字段的顺序需要与每条数据列表中的顺序一致。

In [5]:
def save_mysql(sql, values, **dbinfo):
    conn = pymysql.connect(**dbinfo)
    cursor = conn.cursor()
    try:
        cursor.executemany(sql, values)
        conn.commit()
        print("成功写入", cursor.rowcount, "条数据")
    except Exception as error:
        conn.rollback()
        print("数据写入失败：", error)
    finally:
        cursor.close()
        conn.close()

## 六、运行程序

In [6]:
url = "http://www.bspider.top/doubanbook/"
html = get_html(url)

if html:
    out_list = parser(html)
    print("解析到图书数量：", len(out_list))

    if out_list:
        sql = "INSERT INTO bookinfo(bookname, score, autor, press, pubdate, describ) VALUES(%s, %s, %s, %s, %s, %s)"
        save_mysql(sql, out_list, **dbinfo)

解析到图书数量： 40
成功写入 40 条数据


## 七、查看存储结果

查询数据库中最近写入的几条记录，检查数据是否保存成功。

In [7]:
conn = pymysql.connect(**dbinfo)
cursor = conn.cursor()
cursor.execute("SELECT id, bookname, score, autor, press, pubdate FROM bookinfo ORDER BY id DESC LIMIT 5")
for item in cursor.fetchall():
    print(item)
cursor.close()
conn.close()

{'id': 40, 'bookname': '表现力', 'score': '8.3', 'autor': '[美] 罗布·萨拉菲亚（Rob Salafia）', 'press': '人民邮电出版社', 'pubdate': '2021-6-1'}
{'id': 39, 'bookname': '轴心文明与现代社会', 'score': '9.1', 'autor': '金观涛', 'press': '东方出版社', 'pubdate': '2021-6-1'}
{'id': 38, 'bookname': '情感学习', 'score': '8.7', 'autor': '[德] 乌特·弗雷弗特 等', 'press': '上海人民出版社·光启书局', 'pubdate': '2021-6-30'}
{'id': 37, 'bookname': '夏天只是西瓜做的一个梦', 'score': '8.3', 'autor': '曾仁臻（鱼山饭宽）', 'press': '北京联合出版公司', 'pubdate': '2021-6'}
{'id': 36, 'bookname': '李育辉组织行为学讲义', 'score': '8.7', 'autor': '李育辉', 'press': '新星出版社 | 得到图书', 'pubdate': '2021-6-9'}
